In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

## 라이브러리 불러오기

In [ ]:
import re
import sys
import time
from pathlib import Path

import cv2
import pandas as pd
from paddleocr import PaddleOCR

if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

## OCR 모델 설정 (참고용 - 실제 실행은 병렬 워커 안에서 각자 로딩함)

멀티프로세싱으로 처리하기 때문에, 아래 설정값은 `ocr_worker.py`의 워커 함수 안에
동일하게 들어가 있습니다 (Windows에서 각 워커 프로세스가 독립적으로 모델을 로딩해야
해서, 설정을 노트북과 워커 파일 두 군데에 유지함 - 둘이 다르면 안 됨).

**2단계(1차 빠르게 전체 + 2차 완전미인식만 조건부 재시도) 구조라 모델이 두 개입니다**
(`ocr_worker.py`의 `process_one`/`process_one_sensitive`, `_get_ocr`/`_get_ocr_sensitive`).
`text_det_box_thresh`만 다르고 나머지 설정은 동일합니다.

```python
PaddleOCR(
    lang="korean", device="cpu",
    use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
    enable_mkldnn=False, text_recognition_batch_size=1,
    text_det_box_thresh=0.7,   # 1차: 500장 전체
    # text_det_box_thresh=0.65,  # 2차: 1차 완전미인식만, 시간 여유 있을 때만 재시도
)
```

`text_det_box_thresh` / 재시도 범위 실험 기록 (500장, 2400초 예산 기준):
- 0.7 단독: 1799초, 완전인식 349 / 부분인식 30 / 완전미인식 121
- 0.6 단독(500장 전체): 3243초, 완전인식 361 / 완전미인식 105 (예산 35% 초과)
- 2단계(완전미인식 121장만 0.65 재시도): 1769초, 완전인식 358 / 완전미인식 110
- 2단계(완전미인식+부분인식 151장 0.65 재시도): 2763초, 완전인식 360 / 완전미인식 110
  → 부분인식까지 넓혀도 실제 복구 장수(11장)는 동일했고 시간만 크게 늘어서(예산 초과)
  되돌림. **최종적으로 완전미인식만 재시도.**

**1차 시간 자체의 변동성**: 같은 500장/같은 설정인데도 1386~2120초로 실측 결과가 꽤
달랐음 (로컬 환경 부하에 따라 변함). 그래서 2차를 무조건 실행하지 않고, 1차 소요시간과
재시도 대상 수를 같이 봐서 "이번에 2차까지 하면 예산을 넘길 것 같은지" 미리 추정해서
넘길 것 같으면 2차를 건너뛰는 안전장치를 넣음 (아래 파이프라인 셀 참고).

미인식 원인 분석(105장 중 샘플 조사): 도트프린트(잉크젯 각인) 폰트가 약 50%로 압도적
1위 - 이건 민감도를 낮춰도 구조적으로 잘 안 잡힘. 나머지(저대비/반사, 비결정성 등)가
민감도 조정으로 그나마 커버되는 영역.

## 날짜 후보 추출 정규식

In [ ]:
MONTH_ABBR = r"(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)"

DATE_PATTERNS = [
    re.compile(r"\d{1,4}\s*[.\-/,:]\s*\d{1,2}\s*[.\-/,:]\s*\d{1,4}"),
    re.compile(r"\b\d{1,4}\s+\d{1,2}\s+\d{1,4}\b"),
    re.compile(rf"\d{{1,4}}\s*[./\-,:]?\s*{MONTH_ABBR}\s*[./\-,:]?\s*\d{{1,4}}", re.IGNORECASE),
    re.compile(rf"\b{MONTH_ABBR}\s*[./\-,:]\s*\d{{1,2}}\s*[./\-,:]\s*\d{{2,4}}\b", re.IGNORECASE),
    re.compile(rf"{MONTH_ABBR}\s*\d{{1,2}}\s*\d{{2,4}}", re.IGNORECASE),
    re.compile(r"\b\d{1,2}\s*[.\-/,:]\s*\d{4}\b"),
    re.compile(r"\b\d{4}\s*[.\-/,:]\s*\d{1,2}\b"),
    # YYYY.MMDD (월/일 사이 구분자가 없어서 4자리로 붙은 경우, 예: "2025.0619").
    # 위 패턴(\d{4}[sep]\d{1,2})은 뒷자리를 최대 2자리까지만 잡아서 "2025.0619"의
    # "19"가 후보 밖으로 잘려나갔었음 - 월/일이 붙어있는 4자리 전용 패턴을 따로 둠.
    re.compile(r"\b20\d{2}\s*[.\-/,:]\s*(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\b"),
    # 원래는 \b\d{6,8}\b 로 아무 6~8자리 숫자나 다 잡았음 -> LOT/제품코드 같은 숫자열도
    # 날짜 후보로 오인되는 문제(피드백#5). 연도 자리(20YY 또는 YY, 00~35)로 시작하고
    # 뒤에 유효한 월/일이 이어지는 형태만 허용하도록 좁힘.
    re.compile(r"\b20\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\b"),  # 20YYMMDD (8자리)
    re.compile(r"\b(?:[0-2]\d|3[0-5])(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\b"),  # YYMMDD (6자리)
    re.compile(r"\b20(?:0\d|1\d|2\d|3[0-5])\s+\d{1,2}\s*[\(\[\{]\s*\d{1,2}\b"),
    re.compile(r"\b20\d{2}\s*[:.]\s*\d{1,2}\s*[-./:]\s*\d{1,2}\b"),
    re.compile(r"\b20\d{2}\s+\d{1,2}\s*[.\-/]?\s*\d{1,2}[A-Z]?\b", re.IGNORECASE),
    re.compile(r"\b20\d{2}\s*[.\-/:]\s*\d{4}"),
    re.compile(r"\b\d{1,2}\s*[.\-/]\s*\d{1,2}\s*까지\s*20\d{2}\b"),
]

# 숫자 '5'가 알파벳 'E'로 오독되는 경우가 실제로 관찰됨 (예: "202E1012" -> "20251012").
# itda.ipynb 500장 분석 때 검증된 보정이었는데(000153.jpg 등 복구), 병합 과정에서
# 빠져 있었음. 문맥 전체에서 무작정 E->5로 바꾸면 EXP/DATE 같은 진짜 영어 단어까지
# 깨지므로, "숫자와 E가 섞여 있고 숫자가 하나 이상 있는 짧은 토큰"에서만 좁게 치환함
# (EXP, DATE는 숫자가 하나도 없어서 이 조건에 안 걸림).
_E5_TOKEN = re.compile(r"\b(?=[0-9E]*[0-9])(?=[0-9E]*E)[0-9E]{2,8}\b")

def fix_e5_misread(text):
    return _E5_TOKEN.sub(lambda m: m.group().replace("E", "5"), text)


def find_date_matches(text):
    """텍스트에서 날짜처럼 생긴 후보를 전부 찾아서 (문자열, start, end, 정규식 우선순위)
    로 반환함. 예전엔 여기서 정규식 목록 순서대로 "먼저 매치되면 그 구간을 차지"하는
    식으로 겹치는 후보를 걸러냈는데, 그러면 파싱도 안 되는 후보가 정규식 순서만으로
    진짜 파싱되는 후보의 구간을 선점해버리는 문제가 있었음(피드백). 그래서 여기서는
    겹치든 말든 후보를 전부 모으기만 하고, 실제 "어느 후보가 그 구간을 차지할지"는
    select_best_candidate에서 파싱 성공 여부/점수까지 다 본 다음에 정함."""
    text = str(text)
    fixed_text = fix_e5_misread(text)
    # E->5 보정이 실제로 뭔가 바꿨을 때만 보정된 버전도 같이 스캔함 (같은 길이의
    # 문자열이라 start/end 인덱스가 원본 text에도 그대로 유효함 - 키워드 근접도
    # 계산은 항상 원본 text 기준으로 하면 됨).
    sources = (text,) if fixed_text == text else (text, fixed_text)
    matches = []
    seen = set()
    for source in sources:
        for pattern_idx, pattern in enumerate(DATE_PATTERNS):
            for m in pattern.finditer(source):
                key = (m.start(), m.end(), m.group())
                if key in seen:
                    continue
                seen.add(key)
                matches.append((m.group(), m.start(), m.end(), pattern_idx))
    return matches

## 날짜 필드(연/월/일) 파싱

In [ ]:
MONTH_MAP = {
    "JAN":"01", "FEB":"02", "MAR":"03", "APR":"04",
    "MAY":"05", "JUN":"06", "JUL":"07", "AUG":"08",
    "SEP":"09", "OCT":"10", "NOV":"11", "DEC":"12",
}

def valid_month(x):
    try: return 1 <= int(x) <= 12
    except: return False

def valid_day(x):
    try: return 1 <= int(x) <= 31
    except: return False

def valid_year(x):
    try: return 2000 <= int(x) <= 2035
    except: return False


# OCR이 날짜 뒤에 엉뚱한 숫자를 더 붙이는 경우가 있음
# (예: "2026.08.103" -> 원래는 "2026.08.10"인데 뒤에 "3"이 더 인식됨).
# 이러면 정규식이 "103"을 통째로 한 그룹으로 잡아서 valid_day(103)가 실패하고
# 그냥 NONE으로 버려짐. 3자리 이상이면 앞 2자리만 잘라서 한 번 더 검증해봄
# (숫자가 아예 하나 모자란 경우 - 예: "06.0까지" - 는 복구 불가라 대상 아님).
def safe_month(x):
    if valid_month(x): return x.zfill(2)
    if len(x) > 2 and valid_month(x[:2]): return x[:2]
    return None

def safe_day(x):
    if valid_day(x): return x.zfill(2)
    if len(x) > 2 and valid_day(x[:2]): return x[:2]
    return None


# 3그룹 숫자 중 연도를 못 찾은 애매한 경우, a/c 둘 다 "일(day)" 후보가 됨.
# 이때 둘 중 하나를 무작정 safe_day()로 먼저 시도하면(=자르기부터 시도) 문제가 생김
# (예: "2727.08.18"에서 a="2727"은 사실 깨진 연도인데 뒤 2자리 "27"이 우연히 유효한
# 일자라서 safe_day가 성공해버려, 진짜 일자인 c="18"보다 먼저 채택돼버림).
# 그래서 "자르기 없이 정확히 맞는 후보"를 항상 먼저 확인하고, 그래도 없을 때만
# 자른 값으로 재시도하는 2단계 우선순위를 둠.
def resolve_day(*candidates):
    for c in candidates:
        if valid_day(c): return c.zfill(2)
    for c in candidates:
        if len(c) > 2 and valid_day(c[:2]): return c[:2]
    return None


def parse_fields(candidate):
    s = str(candidate).upper().strip()
    result = {"year": "NONE", "month": "NONE", "day": "NONE"}

    m = re.fullmatch(r"NONE-(\d{2})-(\d{2})", s)
    if m:
        mo = safe_month(m.group(1))
        if mo: result["month"] = mo
        da = safe_day(m.group(2))
        if da: result["day"] = da
        return result

    nums = re.findall(r"\d+", s)

    # 구분자 없이 붙어있는 6~8자리 숫자(예: "20251012", "251012")는 re.findall(\d+)이
    # 통째로 한 덩어리로만 잡아버려서(nums 길이가 1), 아래 3그룹/2그룹 분기 어디에도
    # 안 걸리고 그냥 NONE으로 버려지는 문제가 있었음. DATE_PATTERNS에 이미 20YYMMDD/
    # YYMMDD 전용 패턴이 있으니, 길이 기준으로 직접 잘라서 처리함.
    # 주의: 6자리는 "YYMMDD"로 가정함. 드물게 "(DDMMYY)"처럼 반대 순서를 라벨에 직접
    # 명시하는 수입 제품도 있어서(예: 000379.jpg) 그런 경우는 연/일이 뒤바뀔 수 있음 -
    # 숫자만으로는 두 순서 다 유효하게 통과하는 경우가 있어 일반적으로 구분 불가능함.
    # (다만 텍스트에 "(일월년순)"/"(DDMMYY)" 같은 명시적 힌트가 있으면 아래
    # parse_fields_with_order()가 이 기본 해석을 덮어씀 - select_best_candidate 참고)
    if len(nums) == 1 and len(nums[0]) == 8:
        n = nums[0]
        y, mo_raw, d_raw = n[:4], n[4:6], n[6:8]
        if valid_year(y):
            result["year"] = y
            mo = safe_month(mo_raw)
            if mo: result["month"] = mo
            da = safe_day(d_raw)
            if da: result["day"] = da
        return result

    if len(nums) == 1 and len(nums[0]) == 6:
        n = nums[0]
        y2, mo_raw, d_raw = n[:2], n[2:4], n[4:6]
        if 0 <= int(y2) <= 35:
            result["year"] = "20" + y2
            mo = safe_month(mo_raw)
            if mo: result["month"] = mo
            da = safe_day(d_raw)
            if da: result["day"] = da
        return result

    english_month = None
    for word, month in MONTH_MAP.items():
        if word in s:
            english_month = month
            result["month"] = month
            break

    for n in nums:
        if len(n) == 4 and valid_year(n):
            result["year"] = n
            break

    if english_month and len(nums) >= 2:
        a, b = nums[0], nums[1]
        # 위치(첫번째=일, 두번째=연도) 고정 가정이 "YYYY/MON/DD" 순서에서 틀리는 버그가 있었음
        # (예: "2022/JUN/14" -> a=2022, b=14 였는데 a를 일(day)로 검사해서 실패).
        # 값 기반으로 판별: 4자리+valid_year인 쪽이 연도, 나머지를 일로 확인.
        if len(a) == 4 and valid_year(a):
            result["year"] = a
            da = safe_day(b)
            if da: result["day"] = da
        elif len(b) == 4 and valid_year(b):
            result["year"] = b
            da = safe_day(a)
            if da: result["day"] = da
        else:
            # 둘 다 2자리라 값만으론 구분 불가 -> 기존 관례(첫 숫자=일, 둘째=연도) 유지
            da = safe_day(a)
            if da: result["day"] = da
            if len(b) == 2 and 20 <= int(b) <= 35: result["year"] = "20" + b
        return result

    if len(nums) == 3:
        a, b, c = nums
        # 우선순위: 1) YYYY.MM.DD  2) DD.MM.YYYY  3) YY.MM.DD  4) DD.MM.YY  5) 부분 구제
        # "명확한 4자리 연도"가 항상 "2자리 연도 추정"보다 먼저 오도록 순서를 잡음.
        # 원래는 YY.MM.DD를 DD.MM.YYYY보다 먼저 체크해서, "21.03.2025" 같은 경우
        # a="21"이 2자리라 YY.MM.DD로 잘못 해석(2021년)되면서 뒤에 있는 명백한 4자리
        # 연도 2025를 놓치는 버그가 있었음.
        if len(a) == 4 and valid_year(a):
            # YYYY.MM.DD
            result["year"] = a
            mo = safe_month(b)
            if mo: result["month"] = mo
            da = safe_day(c)
            if da: result["day"] = da
        elif len(c) == 4 and valid_year(c):
            # DD.MM.YYYY
            result["year"] = c
            mo = safe_month(b)
            if mo: result["month"] = mo
            da = safe_day(a)
            if da: result["day"] = da
        elif len(a) == 2 and 20 <= int(a) <= 35:
            # YY.MM.DD - 임의의 "숫자 숫자 숫자" 노이즈가 우연히 걸리는 걸 막기 위해
            # 월이 실제로 유효할 때만 연도까지 같이 채택함(DD.MM.YY와 동일한 안전장치).
            mo = safe_month(b)
            if mo:
                result["year"] = "20" + a
                result["month"] = mo
                da = safe_day(c)
                if da: result["day"] = da
        elif len(c) == 2 and 20 <= int(c) <= 35:
            # DD.MM.YY (연도가 2자리로 맨 뒤에 오는 경우)
            mo = safe_month(b)
            if mo:
                result["year"] = "20" + c
                result["month"] = mo
                da = safe_day(a)
                if da: result["day"] = da
        else:
            mo = safe_month(b)
            if mo:
                result["month"] = mo
                da = resolve_day(a, c)
                if da: result["day"] = da

    elif len(nums) == 2:
        a, b = nums
        if len(b) == 4 and valid_year(b) and valid_month(a):
            result["year"] = b; result["month"] = a.zfill(2)
        elif len(a) == 4 and valid_year(a) and valid_month(b):
            result["year"] = a; result["month"] = b.zfill(2)
        elif len(a) == 4 and valid_year(a) and len(b) == 4 and valid_month(b[:2]) and valid_day(b[2:]):
            # YYYY.MMDD (월/일 사이 구분자가 없어서 하나로 붙은 경우, 예: "2025.0619")
            result["year"] = a
            result["month"] = b[:2]
            result["day"] = b[2:]
        else:
            mo = safe_month(a)
            if mo:
                da = safe_day(b)
                result["month"] = mo
                if da: result["day"] = da

    return result


# 팀원 피드백: 라벨에 "(일월년순)", "(DDMMYY)", "년월일순", "MM/DD/YYYY" 처럼 날짜
# 배열 순서를 텍스트 안에 아예 명시해주는 제품들이 있음. 이런 명시적 힌트가 있을 때는
# 숫자만으로 순서를 추측하는 parse_fields() 대신, 라벨이 알려준 순서(order)를 그대로
# 강제 적용함 - 특히 "050926"처럼 어느 순서로 읽어도 자릿수/범위가 다 그럴듯하게
# 통과해버리는(=parse_fields만으로는 원천적으로 구분 불가능한) 케이스에서 유효함
# (실제로 000379.jpg가 이 케이스: 기본 해석은 "2005-09-26"으로 잘못 나오는데,
# 라벨이 "(일월년순)=DMY"라고 알려주므로 "2026-09-05"가 맞음).
# order가 없거나(=명시적 힌트 없음) order 기준으로도 해석이 안 되면, 항상 기존
# parse_fields()로 안전하게 폴백함 (숫자 추측 로직을 대체하는 게 아니라 우선하는 것).
def parse_fields_with_order(candidate, order):
    s = str(candidate).strip()
    compact = re.sub(r"\D", "", s)
    nums = re.findall(r"\d+", s)

    if order == "DMY":
        if len(compact) == 6:
            d, mo, y = compact[:2], compact[2:4], compact[4:6]
            if valid_day(d) and valid_month(mo) and 0 <= int(y) <= 35:
                return {"year": "20" + y, "month": mo, "day": d}
        if len(compact) == 8:
            d, mo, y = compact[:2], compact[2:4], compact[4:8]
            if valid_day(d) and valid_month(mo) and valid_year(y):
                return {"year": y, "month": mo, "day": d}
        if len(nums) >= 3:
            d, mo, y = nums[0], nums[1], nums[2]
            if valid_day(d) and valid_month(mo):
                if len(y) == 4 and valid_year(y):
                    return {"year": y, "month": mo.zfill(2), "day": d.zfill(2)}
                if len(y) == 2 and 0 <= int(y) <= 35:
                    return {"year": "20" + y, "month": mo.zfill(2), "day": d.zfill(2)}

    elif order == "MDY":
        if len(compact) == 6:
            mo, d, y = compact[:2], compact[2:4], compact[4:6]
            if valid_month(mo) and valid_day(d) and 0 <= int(y) <= 35:
                return {"year": "20" + y, "month": mo, "day": d}
        if len(compact) == 8:
            mo, d, y = compact[:2], compact[2:4], compact[4:8]
            if valid_month(mo) and valid_day(d) and valid_year(y):
                return {"year": y, "month": mo, "day": d}
        if len(nums) >= 3:
            mo, d, y = nums[0], nums[1], nums[2]
            if valid_month(mo) and valid_day(d):
                if len(y) == 4 and valid_year(y):
                    return {"year": y, "month": mo.zfill(2), "day": d.zfill(2)}
                if len(y) == 2 and 0 <= int(y) <= 35:
                    return {"year": "20" + y, "month": mo.zfill(2), "day": d.zfill(2)}

    elif order == "YMD":
        if len(compact) == 8:
            y, mo, d = compact[:4], compact[4:6], compact[6:8]
            if valid_year(y) and valid_month(mo) and valid_day(d):
                return {"year": y, "month": mo, "day": d}
        if len(compact) == 6:
            y, mo, d = compact[:2], compact[2:4], compact[4:6]
            if 0 <= int(y) <= 35 and valid_month(mo) and valid_day(d):
                return {"year": "20" + y, "month": mo, "day": d}
        if len(nums) >= 3:
            y, mo, d = nums[0], nums[1], nums[2]
            if len(y) == 4 and valid_year(y):
                result = {"year": y, "month": "NONE", "day": "NONE"}
                m2 = safe_month(mo)
                if m2: result["month"] = m2
                d2 = safe_day(d)
                if d2: result["day"] = d2
                return result
            if len(y) == 2 and 0 <= int(y) <= 35:
                result = {"year": "20" + y, "month": "NONE", "day": "NONE"}
                m2 = safe_month(mo)
                if m2: result["month"] = m2
                d2 = safe_day(d)
                if d2: result["day"] = d2
                return result

    # 명시 규칙으로 해석 불가능하면 기존 parser로 안전하게 fallback
    return parse_fields(candidate)

## 후보 스코어링 및 최종 선택

In [ ]:
POSITIVE_KEYWORDS = [
    "소비기한", "유통기한", "사용기한",
    "BEST BEFORE", "BEST BY", "BEST IF USED BY",
    "EXP", "EXPIRY", "EXPIRATION", "EXD",
    "BB", "B.B", "까지",
]

NEGATIVE_KEYWORDS = [
    "제조", "제조일", "MFG", "MFD",
    "PACK DATE", "생산", "등급판정일", "LOT",
    # 팀원 피드백: 제조일자 표기의 영문 변형들도 negative로 잡아야 함
    "PROD.DATE", "PROD DATE", "PRODUCTION DATE",
    # 팀원 피드백: "포장일자/포장일/포장일시"도 소비기한이 아닌 제조 계열 날짜라
    # negative로 취급해야 함 (제품에 포장일과 소비기한이 같이 찍히는 경우가 있음)
    "포장일자", "포장일", "포장일시",
]

# 참고: "부터"를 음성 키워드로 추가해서 "A 부터 B 까지" 포맷의 시작일(A)을 직접
# 감점하는 것도 시도해봤는데(000081/000450/000495 등에서는 잘 맞았음), 다른 이미지
# (000453)에서 부터 근처의 진짜 정답 날짜까지 과하게 감점당해서, 텍스트 안의 전혀
# 무관한 후보("2025 1399" - 연도+불량식품신고 전화번호 1399가 우연히 붙어있는 것)가
# 오히려 최고점을 먹어버리는 회귀가 실제로 발생함. "부터~까지" 케이스는 아래
# select_best_candidate의 동점 tie-break(미래 날짜 우선)만으로 이미 다 맞고 있어서
# (500장 검증 완료), 이 감점은 넣지 않기로 함.

# 한글 키워드는 OCR이 (1) 음절 사이에 공백을 끼워넣거나 (예: "소비 기 한")
# (2) 글자 하나를 비슷한 다른 글자로 잘못 읽는(예: 소비기한->소비기안, 등급판정일->등급팔정일)
# 경우가 실제로 관찰됨. 정확 문자열 매칭만 쓰면 이런 오독에서 키워드 자체가
# 무력화되는 문제(피드백#3)라서, 한글 키워드에 한해 "음절 사이 공백 허용 + 글자 1개
# 와일드카드 허용" 정규식을 자동 생성해서 보조로 같이 검사함.
# (영문 키워드는 이런 음절 단위 오독 패턴이 아니라서 대상에서 제외)
#
# 주의: 글자 1개 와일드카드는 키워드가 3자 이하면 사실상 "나머지 2자만" 보고
# 매칭하는 셈이라 너무 느슨해짐. 실제로 "포장일"(3자)을 이 기준(<3자는 제외)으로
# 추가했더니 "포장 년,월,일"처럼 전혀 다른 문구까지 "포장"+와일드카드 1글자로
# 매칭되면서, 그 근처의 진짜 소비기한 후보(2025.12.01, 000407.jpg)를 오히려
# negative로 깎아버리는 회귀가 500장 검증에서 실제로 발견됨. 그래서 와일드카드
# 허용 기준을 4자 미만(<4)으로 올려서, 3자짜리 키워드는 공백 허용만 있는 정확
# 매칭으로 제한함 (기존 4자 이상 키워드들의 동작에는 영향 없음 - 500장 재검증 완료).
def build_fuzzy_keyword_pattern(keyword):
    chars = list(keyword)
    exact = r"\s*".join(re.escape(c) for c in chars)
    if len(chars) < 4:
        # 너무 짧은 키워드(3자 이하)는 글자 하나를 와일드카드로 풀면
        # 아무 문자열에나 매칭될 위험이 커서, 음절 사이 공백 허용만 적용.
        return re.compile(exact)
    variants = [exact]
    for i in range(len(chars)):
        parts = [re.escape(c) if j != i else "." for j, c in enumerate(chars)]
        variants.append(r"\s*".join(parts))
    return re.compile("(?:" + "|".join(variants) + ")")


_HANGUL_ONLY = re.compile(r"^[가-힣]+$")
POSITIVE_KEYWORD_PATTERNS = [build_fuzzy_keyword_pattern(kw) for kw in POSITIVE_KEYWORDS if _HANGUL_ONLY.match(kw)]
NEGATIVE_KEYWORD_PATTERNS = [build_fuzzy_keyword_pattern(kw) for kw in NEGATIVE_KEYWORDS if _HANGUL_ONLY.match(kw)]


def nearest_keyword_distance(text, candidate_start, candidate_end, keywords, keyword_patterns=None):
    text_upper = text.upper()
    best = None
    for kw in keywords:
        kw_upper = kw.upper()
        start = 0
        while True:
            pos = text_upper.find(kw_upper, start)
            if pos == -1:
                break
            kw_end = pos + len(kw_upper)
            distance = min(abs(candidate_start - kw_end), abs(pos - candidate_end))
            if best is None or distance < best:
                best = distance
            start = pos + 1

    if keyword_patterns:
        for pattern in keyword_patterns:
            for m in pattern.finditer(text):
                distance = min(abs(candidate_start - m.end()), abs(m.start() - candidate_end))
                if best is None or distance < best:
                    best = distance

    return best


def score_candidate(text, candidate, start, end):
    score = 0
    nums = re.findall(r"\d+", candidate)

    if len(nums) >= 3: score += 3
    elif len(nums) == 2: score += 1

    for n in nums:
        if len(n) == 4:
            score += 3 if valid_year(n) else -2

    if re.search(MONTH_ABBR, candidate, re.IGNORECASE):
        score += 3

    pos_dist = nearest_keyword_distance(text, start, end, POSITIVE_KEYWORDS, POSITIVE_KEYWORD_PATTERNS)
    neg_dist = nearest_keyword_distance(text, start, end, NEGATIVE_KEYWORDS, NEGATIVE_KEYWORD_PATTERNS)

    if pos_dist is not None:
        if pos_dist <= 10: score += 6
        elif pos_dist <= 25: score += 4
        elif pos_dist <= 50: score += 1

    if neg_dist is not None:
        if neg_dist <= 10: score -= 6
        elif neg_dist <= 25: score -= 4
        elif neg_dist <= 50: score -= 1

    return score


def _date_sort_key(parsed):
    # NONE인 필드는 비교 시 가장 작은 값(-1)으로 취급 -> 값이 채워진 쪽이 우선.
    y = int(parsed["year"]) if parsed["year"] != "NONE" else -1
    m = int(parsed["month"]) if parsed["month"] != "NONE" else -1
    d = int(parsed["day"]) if parsed["day"] != "NONE" else -1
    return (y, m, d)


# 팀원 피드백: "소비기한:읽는 법 (일월년 순)", "(DDMMYY)", "(년월일순)", "MM/DD/YYYY"
# 처럼 라벨에 날짜 배열 순서가 아예 명시된 경우를 잡아내는 패턴. 국가 기반 자동
# 추론(예: 미국산이면 무조건 MDY)은 넣지 않음 - 500장 검증 중 진짜 DD/MM vs MM/DD
# 애매 케이스는 000143.jpg 1건뿐이었고, 그마저도 국가 정보 없이 현재 결과가 문맥상
# 자연스러워서 굳이 자동변환 규칙을 추가할 실익이 없었음. 텍스트 안에 명시적으로
# 순서를 알려주는 경우에만 좁게 적용함.
FORMAT_HINT_PATTERNS = {
    "YMD": [
        re.compile(r"년\s*월\s*일\s*순"),
        re.compile(r"YYYY\s*[/.\-]?\s*MM\s*[/.\-]?\s*DD", re.IGNORECASE),
        re.compile(r"YYYYMMDD", re.IGNORECASE),
    ],
    "DMY": [
        re.compile(r"일\s*월\s*년\s*순"),
        re.compile(r"DD\s*[/.\-]?\s*MM\s*[/.\-]?\s*YYYY", re.IGNORECASE),
        re.compile(r"DDMMYY", re.IGNORECASE),
        re.compile(r"DDMMYYYY", re.IGNORECASE),
    ],
    "MDY": [
        re.compile(r"MM\s*[/.\-]?\s*DD\s*[/.\-]?\s*YYYY", re.IGNORECASE),
        re.compile(r"MMDDYY", re.IGNORECASE),
        re.compile(r"MMDDYYYY", re.IGNORECASE),
    ],
}

def detect_date_order_hint(text):
    text = str(text)
    for order, patterns in FORMAT_HINT_PATTERNS.items():
        for pat in patterns:
            if pat.search(text):
                return order
    return None


def select_best_candidate(text):
    text = str(text)
    # 텍스트에 명시적 배열 힌트가 있으면, 숫자만으로 순서를 추측하는 parse_fields
    # 대신 라벨이 알려준 순서(parse_fields_with_order)를 최우선으로 적용함.
    order_hint = detect_date_order_hint(text)

    raw_matches = find_date_matches(text)
    if not raw_matches:
        return {"year": "NONE", "month": "NONE", "day": "NONE"}

    enriched = []
    for candidate, start, end, pattern_idx in raw_matches:
        parsed = parse_fields_with_order(candidate, order_hint) if order_hint else parse_fields(candidate)
        valid_fields = sum(1 for v in (parsed["year"], parsed["month"], parsed["day"]) if v != "NONE")
        enriched.append({
            "start": start, "end": end, "pattern_idx": pattern_idx,
            "length": end - start,
            "score": score_candidate(text, candidate, start, end),
            "valid_fields": valid_fields,
            "year": parsed["year"], "month": parsed["month"], "day": parsed["day"],
        })

    # overlapping 후보 해소: "파싱된 필드 수(valid_fields) -> 점수(score) -> 길이 ->
    # 텍스트상 위치 -> 정규식 우선순위" 순으로 정렬해서, 이 순서대로 훑으며 이미
    # 차지된 구간과 겹치는 후보는 건너뜀. 원래는 정규식 목록 순서상 먼저 매치된
    # 후보가 무조건 그 구간을 차지해서, 파싱도 안 되는 후보가 진짜 파싱되는(값이
    # 채워지는) 후보를 밀어내는 문제가 있었음.
    enriched.sort(key=lambda x: (-x["valid_fields"], -x["score"], -x["length"], x["start"], x["pattern_idx"]))

    scored = []
    covered = set()
    for c in enriched:
        span = set(range(c["start"], c["end"]))
        if span & covered:
            continue
        covered |= span
        scored.append(c)

    # 스코어만 보고 고르면, 키워드 근처에 있지만 실제로는 파싱이 안 되는(전부 NONE)
    # 후보가 점수만 높아서 진짜 파싱되는 후보를 밀어내고 선택될 수 있음 - 파싱 성공한
    # 후보가 하나라도 있으면 그쪽에서만 고르도록 필터링.
    parseable = [c for c in scored if c["valid_fields"] > 0]
    if parseable:
        scored = parseable
    else:
        return {"year": "NONE", "month": "NONE", "day": "NONE"}

    # 점수가 동점(키워드 신호가 없거나 여러 후보가 똑같이 애매)일 때, 원래는 그냥
    # find_date_matches가 찾은 순서상 "먼저 나온 후보"가 이겼음. 하지만 소비/유통기한은
    # 거의 항상 텍스트상의 다른 날짜(제조일 등)보다 미래이므로, 예전 파이프라인
    # (itda.ipynb STEP C+D, 500장 중 240장이 이 규칙에 의존)처럼 동점일 때는
    # "연도/월/일이 가장 큰(미래) 후보"를 최종 채택하도록 함.
    max_score = max(c["score"] for c in scored)
    top = [c for c in scored if c["score"] == max_score]
    best = max(top, key=_date_sort_key) if len(top) > 1 else top[0]
    return {"year": best["year"], "month": best["month"], "day": best["day"]}

## 회귀 테스트 (지금까지 발견된 버그 케이스 고정)

지금까지 발견/수정한 버그들이 나중에 로직을 또 고치다가 다시 깨지지 않도록,
확인된 케이스들을 assert로 고정해둠. 실패하면 어디서 회귀가 생겼는지 바로 알 수 있음.

In [ ]:
# 피드백#1: 영어 월 표기에서 연도가 먼저 나오는 순서 ("YYYY/MON/DD")
assert parse_fields("2022/JUN/14") == {"year": "2022", "month": "06", "day": "14"}

# 피드백#5: LOT/제품코드 같은 임의의 6~8자리 숫자는 날짜 후보로 잡히면 안 됨
assert find_date_matches("LOT 930249") == []

# 피드백#3: 한글 키워드 오독(음절 사이 공백 삽입 / 글자 1개 오독) 대응
assert any(p.search("소 비 기 한") for p in POSITIVE_KEYWORD_PATTERNS)
assert any(p.search("소비기안") for p in POSITIVE_KEYWORD_PATTERNS)
assert any(p.search("등급팔정일") for p in NEGATIVE_KEYWORD_PATTERNS)

# 동점 tie-break: "A 부터 B 까지" 포맷에서 미래(종료) 날짜를 선택해야 함
assert select_best_candidate("소비기한 부터 2025.05.19 2026.05.18 까지") == {"year": "2026", "month": "05", "day": "18"}

# 잘린/오염된 숫자 복구: 날짜 뒤에 엉뚱한 숫자가 더 붙은 경우 앞 2자리로 복구
assert parse_fields("2026.08.103") == {"year": "2026", "month": "08", "day": "10"}

# 위 복구 로직이 "정확히 맞는 후보"보다 먼저 나서면 안 됨
# ("2727.08.18"에서 a="2727"은 깨진 연도인데 뒤 2자리 "27"이 우연히 유효한 일자라서
#  자르기부터 시도하면 진짜 일자인 "18"을 밀어내는 회귀가 실제로 있었음)
assert parse_fields("2727.08.18") == {"year": "NONE", "month": "08", "day": "18"}

# E->5 오독 보정: "202E1012" -> "20251012" (itda.ipynb에서 검증된 케이스, 000153.jpg)
assert any(c[0] == "20251012" for c in find_date_matches("EXP:202E1012"))

# 구분자 없이 붙어있는 6~8자리 숫자는 re.findall(\d+)이 통째로 한 덩어리로만 잡아서
# 3그룹/2그룹 분기 어디에도 안 걸리고 버려지는 문제가 있었음 (길이 기준으로 직접 분해)
assert parse_fields("20251012") == {"year": "2025", "month": "10", "day": "12"}

# DD.MM.YY (연도가 2자리로 맨 뒤에 오는 경우) - 이전엔 팀원 원본 로직에도 없던 케이스라
# year가 그냥 NONE으로 버려지고 있었음
assert parse_fields("05.09.26") == {"year": "2026", "month": "09", "day": "05"}

# 위 케이스 추가하면서 생긴 회귀: "9 85 27" 같은 무관한 숫자 나열이 85를 월로 잘못
# 받아들이면서 연도(27)만 보고 확정해버리는 문제 -> 월도 유효해야만 채택하도록 함
assert parse_fields("9 85 27") == {"year": "NONE", "month": "NONE", "day": "NONE"}

# 콜론(:)도 구분자로 인정 (itda.ipynb V5 정규식에 있었는데 병합 과정에서 빠졌던 부분)
assert any(c[0] == "2025:10:12" for c in find_date_matches("EXP2025:10:12"))

# ===== 팀원이 보내준 개선사항 5개 반영 =====

# 팀원#1: YYYY.MMDD (월/일 사이 구분자가 없어서 4자리로 붙은 경우). parse_fields만
# 고치는 걸로는 부족했음 - 정규식 목록에도 "4자리+구분자+4자리"를 통째로 잡는 패턴이
# 없어서 후보 자체가 안 만들어졌었음(뒷자리가 2자리까지만 잘려서 잡혔었음).
assert parse_fields("2025.0619") == {"year": "2025", "month": "06", "day": "19"}
assert select_best_candidate("소비기한 2025.0619") == {"year": "2025", "month": "06", "day": "19"}

# 팀원#4: 3그룹 파싱에서 "명확한 4자리 연도"가 "2자리 연도 추정"보다 항상 우선해야 함
# ("21.03.2025"에서 a="21"이 2자리라 YY.MM.DD로 잘못 해석(2021년)하면서 뒤에 있는
#  명백한 4자리 연도 2025를 놓치는 버그가 있었음 - DD.MM.YYYY를 YY.MM.DD보다 먼저 체크)
assert parse_fields("21.03.2025") == {"year": "2025", "month": "03", "day": "21"}

# 팀원#2: 점수만 보고 고르면 안 됨 - 키워드 바로 옆에 있지만 실제로는 파싱이 안 되는
# (전부 NONE) 후보가 점수만 높아서, 좀 더 떨어져 있어도 실제로 파싱되는 후보를
# 밀어내면 안 됨
assert select_best_candidate("소비기한 99 99 99 2026.03.15") == {"year": "2026", "month": "03", "day": "15"}

# 팀원#5: overlapping 후보 처리 - "2026 13 40"은 월(13)/일(40)이 둘 다 무효라 연도만
# 부분 채워짐. 정규식 우선순위가 아니라 파싱 결과 기준으로 후보가 선택돼야 함.
assert select_best_candidate("2026 13 40") == {"year": "2026", "month": "NONE", "day": "NONE"}

# ===== 팀원이 추가로 보내준 개선사항 (명시적 배열 힌트 / negative 키워드 보강) =====

# 명시적 배열 힌트 탐지: "(일월년 순)"/"(DDMMYY)" -> DMY, "(년월일순)"/"YYYYMMDD" -> YMD,
# "MM/DD/YYYY" -> MDY. 배열규칙만들기.ipynb에서 실측 500장 중 3장(000205/000379/000449)
# 에서만 힌트가 발견됐고, 그중 000379 1장만 최종 결과가 바뀌었음(아래 참고).
assert detect_date_order_hint("소비기한:읽는 법 (일월년 순)BEST BEFORE: (DDMMYY) 050926") == "DMY"
assert detect_date_order_hint("유통기한 (년월일순) 20260905") == "YMD"
assert detect_date_order_hint("EXP MM/DD/YYYY 09/05/2026") == "MDY"
assert detect_date_order_hint("힌트 없는 일반 텍스트 2026.09.05") is None

# 실제 000379.jpg 케이스: "050926"은 자릿수/범위만으로는 YYMMDD로도 DDMMYY로도 둘 다
# 그럴듯하게 통과해버려서(=parse_fields만으로는 원천적으로 구분 불가능), 힌트 없이는
# "2005-09-26"으로 잘못 해석됐었음. 라벨이 "(일월년순)=DMY"라고 명시했으므로
# "2026-09-05"가 정답 - 배열규칙만들기.ipynb에서 팀원이 직접 확인한 케이스와 동일함.
assert parse_fields_with_order("050926", "DMY") == {"year": "2026", "month": "09", "day": "05"}
assert select_best_candidate("소비기한:읽는 법 (일월년 순)BEST BEFORE: (DDMMYY) 050926") == {"year": "2026", "month": "09", "day": "05"}

# negative 키워드 보강: "PRODUCTION DATE"/"포장일자" 근처 날짜는 감점돼야 함.
# 단, "포장일"(3자)처럼 짧은 한글 키워드에 글자 1개 와일드카드 퍼지매칭을 그대로
# 적용하면 "포장"+아무글자 1개로 사실상 다 걸려버려서(예: "포장 년,월,일" 같은
# 무관한 문구까지 매칭), 그 옆의 진짜 소비기한 날짜까지 오염되는 회귀가 500장
# 재검증에서 실제로 발견됨(000407.jpg: 정답 2025.12.01이 밀려남) - 그래서
# 와일드카드 허용 기준을 4자 이상으로 올렸고, "포장일"은 공백 허용 정확매칭만 적용됨.
assert select_best_candidate("PRODUCTION DATE 2025.01.01 소비기한 2026.03.15") == {"year": "2026", "month": "03", "day": "15"}
assert select_best_candidate("포장일자 2025.01.01 소비기한 2026.03.15") == {"year": "2026", "month": "03", "day": "15"}
# "포장" + 전혀 다른 문구는 "포장일"에 오매칭되면 안 됨 (000407.jpg 회귀 재발 방지)
assert not any(p.search("포장 년,월,일") for p in NEGATIVE_KEYWORD_PATTERNS)

print("회귀 테스트 전부 통과")

## 전체 이미지 처리 (멀티프로세싱, 2단계 + 시간 안전장치)

CPU 코어 개수만큼 병렬로 OCR을 돌립니다 (`mp.cpu_count()`로 자동 감지 - 로컬 2코어,
채점 서버 4코어). OCR만 워커에서 병렬 처리하고, 날짜 후보 추출/파싱/스코어링은
빠른 연산이라 결과를 모은 뒤 메인 프로세스에서 처리합니다.

**1차**: `box_thresh=0.7`로 전체 이미지 처리. **2차**: 1차 결과가 완전미인식(NONE)이었던
이미지만 추려서 `box_thresh=0.65`로 재시도 - 단, "1차 소요시간 + 재시도 대상 수 × 장당
예상시간"이 예산(2400초) - 안전마진(200초)을 넘을 것 같으면 2차를 건너뜁니다 (1차 시간
자체가 실측에서 1386~2120초로 들쭉날쭉해서, 시간만 보고 판단하면 위험함 - 재시도 대상
수까지 같이 봐야 함). 2차 결과는 1차보다 채워진 필드 수가 늘어났을 때만 덮어씁니다.

In [ ]:
import multiprocessing as mp
import ocr_worker

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
input_path = Path(INPUT_DIR)
image_files = sorted(p for p in input_path.iterdir() if p.suffix.lower() in IMAGE_EXTS) if input_path.exists() else []

# 워커 개수: 환경변수로 직접 지정 가능 (안 정해주면 mp.cpu_count() 자동감지).
# 로컬처럼 물리 코어보다 논리 코어가 부풀려 보이는 환경에서는 직접 낮춰서
# 테스트하는 게 나을 수 있음 (예: ITDA_N_WORKERS=2). 채점 서버에서는 안 건드리면 됨.
N_WORKERS = int(os.environ.get("ITDA_N_WORKERS", mp.cpu_count()))

if __name__ == "__main__":
    t_start = time.time()
    image_paths = [str(p) for p in image_files]

    # ===== 1차: box_thresh=0.7 로 전체 이미지 처리 (빠름) =====
    with mp.Pool(processes=N_WORKERS) as pool:
        ocr_results = pool.map(ocr_worker.process_one, image_paths)

    rows = []
    for image_id, text, seconds in ocr_results:
        fields = select_best_candidate(text)
        year, month, day = fields["year"], fields["month"], fields["day"]
        final_date = f"{year}-{month}-{day}" if not (year == "NONE" and month == "NONE" and day == "NONE") else "NONE"
        rows.append({
            "image_id": image_id,
            "year": year, "month": month, "day": day,
            "final_date": final_date,
        })

    t_pass1 = time.time() - t_start
    n_none_pass1 = sum(1 for r in rows if r["final_date"] == "NONE")
    # 2차 재시도 대상: 완전미인식(NONE)만. 부분인식까지 넓혀서 실측해봤는데
    # (완전미인식121+부분인식30=151장 재시도, 642초 추가) 회복량은 거의 그대로인 채
    # 시간만 크게 늘어서(2763초, 2400초 예산 초과) 되돌림 - 완전미인식만으로도
    # 실제 복구되는 장수(11장)는 동일하게 나왔음.
    # image_id는 확장자 제외 파일명(ocr_worker._run 참고)이라, 원본 파일 목록과
    # 매칭할 때도 p.name이 아니라 p.stem으로 비교해야 함.
    retry_ids = {r["image_id"] for r in rows if r["final_date"] == "NONE"}
    retry_paths = [str(p) for p in image_files if p.stem in retry_ids]
    print(f"1차(민감도 0.7) 완료: 워커 {N_WORKERS}개, 총 {len(rows)}장, {t_pass1:.0f}초, 완전미인식 {n_none_pass1}장 -> 2차 재시도 대상")

## 2차 재시도 (완전미인식만 민감도 0.65)

1차 셀에서 만든 `rows`/`retry_paths`를 그대로 이어받아서 씁니다 (1차를 다시 돌릴 필요 없음).

`importlib.reload(ocr_worker)`를 넣어둔 이유: 노트북 커널이 이 세션 안에서 `ocr_worker`를
이미 한 번 import한 뒤에 `ocr_worker.py` 파일이 수정되면(예: `process_one_sensitive` 함수
추가), 커널은 그 변경을 모르고 예전 버전을 계속 메모리에 캐싱해서 `AttributeError`가 남 -
reload로 강제로 다시 읽어오게 함.

In [ ]:
import importlib
importlib.reload(ocr_worker)

def _filled_count(year, month, day):
    return sum(1 for v in (year, month, day) if v != "NONE")

# 시간 안전장치: 1차 시간 자체가 실측에서 1386~2120초로 많이 들쭉날쭉했음
# (같은 500장, 같은 설정인데도 53% 차이). 그래서 "1차가 얼마나 걸렸는지"뿐 아니라
# "2차 대상이 몇 장인지"까지 같이 봐서, 2차를 실제로 돌렸을 때 예상 총 시간이
# 예산(2400초)에서 안전마진을 넘지 않을 때만 2차를 진행함.
# 장당 예상시간(4.0초)은 완전미인식만 재시도했을 때 실측치(3.17초/장)에 약간의
# 여유만 더한 값 - 처음엔 4.5초+200초 마진으로 이중으로 보수적이게 잡아서, 실제로는
# 충분히 안전한 케이스(1차 1686초+121장)까지 걸러버리는 일이 있었음. 그래서 마진을
# 150초로, 장당 예상시간도 좀 더 실측에 가깝게 낮춤.
BUDGET_SECONDS = 2400
SAFETY_MARGIN = 150
SENSITIVE_SEC_PER_IMAGE = 4.0

est_pass2 = len(retry_paths) * SENSITIVE_SEC_PER_IMAGE
should_run_pass2 = retry_paths and (t_pass1 + est_pass2 <= BUDGET_SECONDS - SAFETY_MARGIN)

t_pass2 = 0.0
n_recovered = 0   # 완전미인식(NONE) -> 뭔가 채워진 경우
if should_run_pass2:
    t_pass2_start = time.time()
    with mp.Pool(processes=N_WORKERS) as pool:
        retry_results = pool.map(ocr_worker.process_one_sensitive, retry_paths)
    t_pass2 = time.time() - t_pass2_start

    retry_text_by_id = {image_id: text for image_id, text, seconds in retry_results}
    for row in rows:
        if row["image_id"] in retry_text_by_id:
            before_filled = _filled_count(row["year"], row["month"], row["day"])
            fields = select_best_candidate(retry_text_by_id[row["image_id"]])
            year, month, day = fields["year"], fields["month"], fields["day"]
            after_filled = _filled_count(year, month, day)

            # 2차 결과가 1차보다 "채워진 필드 수"가 늘어났을 때만 덮어씀
            # (완전미인식은 채워진 게 0개라 조금만 나아져도 무조건 이득, 비결정성
            # 때문에 더 나빠지는 경우는 없음 - 그래도 일관성 있게 같은 조건 적용).
            if after_filled > before_filled:
                final_date = f"{year}-{month}-{day}" if not (year == "NONE" and month == "NONE" and day == "NONE") else "NONE"
                row["year"], row["month"], row["day"], row["final_date"] = year, month, day, final_date
                if final_date != "NONE":
                    n_recovered += 1

    print(f"2차(민감도 0.65) 완료: {len(retry_paths)}장 재시도, {t_pass2:.0f}초, {n_recovered}장 복구")
elif not retry_paths:
    print("1차에서 완전미인식이 없어서 2차 생략")
else:
    print(f"2차 생략: 1차가 {t_pass1:.0f}초 걸렸고 재시도 대상이 {len(retry_paths)}장이라, "
          f"예상 총시간({t_pass1 + est_pass2:.0f}초)이 안전마진(예산 {BUDGET_SECONDS}초 - "
          f"{SAFETY_MARGIN}초)을 넘어서 시간 초과 위험 때문에 건너뜀")

df = pd.DataFrame(rows)
elapsed = t_pass1 + t_pass2
print(f"\n전체 완료: 총 {len(df)}장, {elapsed:.0f}초 (1차 {t_pass1:.0f}초 + 2차 {t_pass2:.0f}초, 장당 평균 {elapsed/max(len(df),1):.2f}초)")

## 결과 저장

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

n_full = ((df['year']!='NONE')&(df['month']!='NONE')&(df['day']!='NONE')).sum()
n_none = (df['final_date']=='NONE').sum()
n_partial = len(df) - n_full - n_none

print(f"저장 완료: {OUTPUT_PATH} ({len(df)}장)")
print(f"완전 인식(연/월/일 전부): {n_full}장")
print(f"부분 인식(일부만 채워짐): {n_partial}장")
print(f"완전 미인식(NONE): {n_none}장")